In [5]:
"""
PYSTAC-Monty CSV Extractor

This script extracts data from PYSTAC-Monty transformers.
Requires pystac-monty libraries to be installed and accessible.
"""

import argparse
import csv
import json
import logging
import os
import sys
from typing import Dict, List, Any, Optional, Tuple, Type

# Configure logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(name)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

# Define a function to check and import required modules
def import_pystac_monty_modules():
    # Check if pystac is installed
    try:
        import pystac
    except ImportError:
        logger.error("Failed to import pystac module")
        logger.error("Please make sure pystac library is installed")
        sys.exit(1)
        
    # Check if pystac_monty is installed
    try:
        import pystac_monty
    except ImportError:
        logger.error("Failed to import pystac_monty module")
        logger.error("Please make sure pystac-monty library is installed")
        sys.exit(1)
    
    # Now import all required modules
    global Item, MontyDataTransformer, MontyDataSource, MontyGeoCoder, MontyExtension, MontyHazardProfiles
    global EMDATTransformer, EMDATDataSource, DesinventarTransformer, DesinventarDataSource
    global GDACSTransformer, GDACSDataSource, GDACSDataSourceType, GFDTransformer, GFDDataSource
    global GIDDTransformer, GIDDDataSource, GlideTransformer, GlideDataSource
    global IBTrACSTransformer, IBTrACSDataSource, IDUTransformer, IDUDataSource
    global PDCTransformer, PDCDataSource, USGSTransformer, USGSDataSource
    
    from pystac import Item
    from pystac_monty.sources.common import MontyDataTransformer, MontyDataSource
    from pystac_monty.geocoding import MontyGeoCoder
    from pystac_monty.extension import MontyExtension
    from pystac_monty.hazard_profiles import MontyHazardProfiles

    # Import all transformers and data sources
    from pystac_monty.sources.emdat import EMDATTransformer, EMDATDataSource
    from pystac_monty.sources.desinventar import DesinventarTransformer, DesinventarDataSource
    from pystac_monty.sources.gdacs import GDACSTransformer, GDACSDataSource
    from pystac_monty.sources.gfd import GFDTransformer, GFDDataSource
    from pystac_monty.sources.gidd import GIDDTransformer, GIDDDataSource
    from pystac_monty.sources.glide import GlideTransformer, GlideDataSource
    from pystac_monty.sources.ibtracs import IBTrACSTransformer, IBTrACSDataSource
    from pystac_monty.sources.idu import IDUTransformer, IDUDataSource
    #from pystac_monty.sources.ifrcevent import IFRCEventTransformer, IFRCEventDataSource
    from pystac_monty.sources.pdc import PDCTransformer, PDCDataSource
    from pystac_monty.sources.usgs import USGSTransformer, USGSDataSource

# Call the import function
import_pystac_monty_modules()


class StacItemExtractor:
    """Extract data from STAC Items produced by MontyDataTransformers."""
    
    def __init__(self, output_dir: str = "output"):
        """Initialize the extractor.
        
        Args:
            output_dir: Directory where CSV files will be saved
        """
        self.output_dir = output_dir
        os.makedirs(output_dir, exist_ok=True)
        
        # Initialize data containers
        self.events: List[Dict[str, Any]] = []
        self.hazards: List[Dict[str, Any]] = []
        self.impacts: List[Dict[str, Any]] = []
        
        # Track relationships between items
        self.event_map: Dict[str, str] = {}  # Maps hazard/impact to event
        self.hazard_map: Dict[str, str] = {}  # Maps impact to hazard
    
    def extract_from_transformer(self, transformer: MontyDataTransformer, source_name: str) -> None:
        """Extract data from a transformer and add it to the collection.
        
        Args:
            transformer: The MontyDataTransformer to extract data from
            source_name: Name of the data source
        """
        logger.info(f"Extracting data from {source_name} transformer")
        
        # Process items from the transformer
        current_event_id = None
        current_hazard_id = None
        
        for item in transformer.get_stac_items():
            roles = item.properties.get("roles", [])
            
            if "event" in roles:
                # Store event and set as current event
                event_data = self._extract_event_data(item, source_name)
                self.events.append(event_data)
                current_event_id = item.id
                
            elif "hazard" in roles:
                # Store hazard and link to current event
                if current_event_id:
                    self.event_map[item.id] = current_event_id
                
                hazard_data = self._extract_hazard_data(item, source_name)
                self.hazards.append(hazard_data)
                current_hazard_id = item.id
                
            elif "impact" in roles:
                # Store impact and link to current event and hazard
                if current_event_id:
                    self.event_map[item.id] = current_event_id
                if current_hazard_id:
                    self.hazard_map[item.id] = current_hazard_id
                
                impact_data = self._extract_impact_data(item, source_name)
                self.impacts.append(impact_data)
    
    def _extract_event_data(self, item: Item, source: str) -> Dict[str, Any]:
        """Extract data from an event item.
        
        Args:
            item: STAC Item representing an event
            source: Name of the data source
            
        Returns:
            Dictionary of extracted event data
        """
        # Extract MontyExtension data
        monty = MontyExtension.ext(item)
        country_codes = monty.country_codes if hasattr(monty, "country_codes") else []
        hazard_codes = monty.hazard_codes if hasattr(monty, "hazard_codes") else []
        correlation_id = monty.correlation_id if hasattr(monty, "correlation_id") else None
        
        # Get start and end datetimes
        start_datetime = item.properties.get("start_datetime", None)
        end_datetime = item.properties.get("end_datetime", None)
        
        # Extract event data
        event_data = {
            "id": item.id,
            "source": source,
            "title": item.properties.get("title", ""),
            "description": item.properties.get("description", ""),
            "datetime": item.datetime.isoformat() if item.datetime else None,
            "start_datetime": start_datetime,
            "end_datetime": end_datetime,
            "country_codes": "|".join(country_codes) if country_codes else "",
            "hazard_codes": "|".join(hazard_codes) if hazard_codes else "",
            "correlation_id": correlation_id,
            "bbox": json.dumps(item.bbox) if item.bbox else "",
            "geometry": json.dumps(item.geometry) if item.geometry else ""
        }
        
        return event_data
    
    def _extract_hazard_data(self, item: Item, source: str) -> Dict[str, Any]:
        """Extract data from a hazard item.
        
        Args:
            item: STAC Item representing a hazard
            source: Name of the data source
            
        Returns:
            Dictionary of extracted hazard data
        """
        # Extract MontyExtension data
        monty = MontyExtension.ext(item)
        country_codes = monty.country_codes if hasattr(monty, "country_codes") else []
        hazard_codes = monty.hazard_codes if hasattr(monty, "hazard_codes") else []
        correlation_id = monty.correlation_id if hasattr(monty, "correlation_id") else None
        
        # Extract hazard detail
        hazard_detail = monty.hazard_detail if hasattr(monty, "hazard_detail") else None
        severity_value = None
        severity_unit = None
        severity_label = None
        if hazard_detail:
            severity_value = hazard_detail.severity_value if hasattr(hazard_detail, "severity_value") else None
            severity_unit = hazard_detail.severity_unit if hasattr(hazard_detail, "severity_unit") else None
            severity_label = hazard_detail.severity_label if hasattr(hazard_detail, "severity_label") else None
        
        # Get linked event_id
        event_id = self.event_map.get(item.id, "")
        
        # Extract hazard data
        hazard_data = {
            "id": item.id,
            "event_id": event_id,
            "source": source,
            "title": item.properties.get("title", ""),
            "description": item.properties.get("description", ""),
            "datetime": item.datetime.isoformat() if item.datetime else None,
            "country_codes": "|".join(country_codes) if country_codes else "",
            "hazard_codes": "|".join(hazard_codes) if hazard_codes else "",
            "correlation_id": correlation_id,
            "severity_value": severity_value,
            "severity_unit": severity_unit,
            "severity_label": severity_label,
            "bbox": json.dumps(item.bbox) if item.bbox else "",
            "geometry": json.dumps(item.geometry) if item.geometry else ""
        }
        
        return hazard_data
    
    def _extract_impact_data(self, item: Item, source: str) -> Dict[str, Any]:
        """Extract data from an impact item.
        
        Args:
            item: STAC Item representing an impact
            source: Name of the data source
            
        Returns:
            Dictionary of extracted impact data
        """
        # Extract MontyExtension data
        monty = MontyExtension.ext(item)
        country_codes = monty.country_codes if hasattr(monty, "country_codes") else []
        correlation_id = monty.correlation_id if hasattr(monty, "correlation_id") else None
        
        # Extract impact detail
        impact_detail = monty.impact_detail if hasattr(monty, "impact_detail") else None
        impact_category = None
        impact_type = None
        value = None
        unit = None
        if impact_detail:
            impact_category = impact_detail.category if hasattr(impact_detail, "category") else None
            impact_type = impact_detail.type if hasattr(impact_detail, "type") else None
            value = impact_detail.value if hasattr(impact_detail, "value") else None
            unit = impact_detail.unit if hasattr(impact_detail, "unit") else None
        
        # Get linked event_id and hazard_id
        event_id = self.event_map.get(item.id, "")
        hazard_id = self.hazard_map.get(item.id, "")
        
        # Extract impact data
        impact_data = {
            "id": item.id,
            "event_id": event_id,
            "hazard_id": hazard_id,
            "source": source,
            "title": item.properties.get("title", ""),
            "description": item.properties.get("description", ""),
            "datetime": item.datetime.isoformat() if item.datetime else None,
            "country_codes": "|".join(country_codes) if country_codes else "",
            "correlation_id": correlation_id,
            "impact_category": str(impact_category) if impact_category else "",
            "impact_type": str(impact_type) if impact_type else "",
            "value": value,
            "unit": unit,
            "bbox": json.dumps(item.bbox) if item.bbox else "",
            "geometry": json.dumps(item.geometry) if item.geometry else ""
        }
        
        return impact_data
    
    def save_to_csv(self) -> None:
        """Save extracted data to CSV files."""
        # Save events
        if self.events:
            self._save_list_to_csv(self.events, os.path.join(self.output_dir, "events.csv"))
        
        # Save hazards
        if self.hazards:
            self._save_list_to_csv(self.hazards, os.path.join(self.output_dir, "hazards.csv"))
        
        # Save impacts
        if self.impacts:
            self._save_list_to_csv(self.impacts, os.path.join(self.output_dir, "impacts.csv"))
    
    def _save_list_to_csv(self, data_list: List[Dict[str, Any]], output_path: str) -> None:
        """Save a list of dictionaries to a CSV file.
        
        Args:
            data_list: List of dictionaries to save
            output_path: Path to save the CSV file
        """
        if not data_list:
            logger.warning(f"No data to save to {output_path}")
            return
        
        # Get fieldnames from the first item
        fieldnames = list(data_list[0].keys())
        
        logger.info(f"Saving {len(data_list)} records to {output_path}")
        
        with open(output_path, 'w', newline='', encoding='utf-8') as csvfile:
            writer = csv.DictWriter(csvfile, fieldnames=fieldnames)
            writer.writeheader()
            writer.writerows(data_list)


# Define mock data configurations for each source with JSON strings
SOURCE_CONFIGS = {
    "emdat": {
        "source_url": "https://public.emdat.be/data",
        "data": json.dumps({"query": "select * from emdat limit 10"})  # Convert dict to JSON string
    },
    "desinventar": {
        "tmp_zip_file": "data/temp/desinventar.zip",
        "country_code": "col",  # Colombia
        "iso3": "COL"
    },
    "gdacs_flood": {
        "source_url": "https://www.gdacs.org/gdacsapi/api/events",
        "data": json.dumps({"eventtype": "FL", "eventid": "1102983"}),  # Convert dict to JSON string
        "type": "FL"  # Use string "FL" for flood events
    },
    "gdacs_drought": {
        "source_url": "https://www.gdacs.org/gdacsapi/api/events",
        "data": json.dumps({"eventtype": "DR", "eventid": ""}),  # Convert dict to JSON string
        "type": "DR"  # Use string "DR" for drought events
    },
    "gfd": {
        "source_url": "https://global-flood-database.cloudtostreet.ai/",
        "data": json.dumps({"region": "global", "limit": 10})  # Convert dict to JSON string
    },
    "gidd": {
        "source_url": "https://helix-tools-api.idmcdb.org/external-api/gidd/disaggregations/disaggregation-geojson/",
        "data": json.dumps({"client_id": "demo"})  # Convert dict to JSON string
    },
    "glide": {
        "source_url": "https://www.glidenumber.net/glide/jsonglideset.jsp",
        "data": json.dumps({"level1": "ALL", "fromyear": "2023", "toyear": "2025"})  # Convert dict to JSON string
    },
    "ibtracs": {
        "source_url": "https://www.ncei.noaa.gov/data/international-best-track-archive-for-climate-stewardship-ibtracs/v04r01/access/csv/",
        "data": json.dumps({"basin": "NA", "year": "2024"})  # Convert dict to JSON string
    },
    "idu": {
        "source_url": "https://helix-tools-api.idmcdb.org/external-api/",
        "data": json.dumps({"client_id": "demo"})  # Convert dict to JSON string
    },
    "pdc": {
        "source_url": "https://sentry.pdc.org/hp_srv/services/hazards/",
        "data": json.dumps({"limit": 10})  # Convert dict to JSON string
    },
    "usgs": {
        "source_url": "https://earthquake.usgs.gov/earthquakes/feed/v1.0/summary/",
        "data": json.dumps({"period": "day"})  # Convert dict to JSON string
    }
}


# Define mapping of source names to transformer/data source classes
SOURCE_MAPPING = {
    "emdat": (EMDATTransformer, EMDATDataSource),
    "desinventar": (DesinventarTransformer, DesinventarDataSource),
    "gdacs_flood": (GDACSTransformer, GDACSDataSource),
    "gdacs_drought": (GDACSTransformer, GDACSDataSource),
    "gfd": (GFDTransformer, GFDDataSource),
    "gidd": (GIDDTransformer, GIDDDataSource),
    "glide": (GlideTransformer, GlideDataSource),
    "ibtracs": (IBTrACSTransformer, IBTrACSDataSource),
    "idu": (IDUTransformer, IDUDataSource),
    #"ifrcevent": (IFRCEventTransformer, IFRCEventDataSource),
    "pdc": (PDCTransformer, PDCDataSource),
    "usgs": (USGSTransformer, USGSDataSource),
}


def setup_logging(level: int = logging.INFO, log_file: Optional[str] = None) -> None:
    """Set up logging configuration.
    
    Args:
        level: Logging level (default: logging.INFO)
        log_file: Path to log file (if None, log to stdout only)
    """
    handlers = [logging.StreamHandler(sys.stdout)]
    
    if log_file:
        handlers.append(logging.FileHandler(log_file))
    
    logging.basicConfig(
        level=level,
        format='%(asctime)s - %(name)s - %(levelname)s - %(message)s',
        handlers=handlers
    )


def create_parser() -> argparse.ArgumentParser:
    """Create an argument parser for the CLI.
    
    Returns:
        An argument parser for the CLI
    """
    parser = argparse.ArgumentParser(
        description="Extract data from PYSTAC-Monty transformers and output as CSV"
    )
    
    subparsers = parser.add_subparsers(dest="command", help="Command to run")
    
    # Extract command
    extract_parser = subparsers.add_parser("extract", help="Extract data from transformers")
    extract_parser.add_argument(
        "--source", "-s", 
        required=True,
        help="Source(s) to extract data from, comma-separated if multiple"
    )
    extract_parser.add_argument(
        "--output", "-o",
        default="output",
        help="Directory to save output CSV files (default: output)"
    )
    extract_parser.add_argument(
        "--verbose", "-v",
        action="store_true",
        help="Enable verbose logging"
    )
    extract_parser.add_argument(
        "--log-file",
        help="Path to save log file"
    )
    
    # List command (to list available sources)
    list_parser = subparsers.add_parser("list", help="List available sources")
    
    return parser


def extract_command(
    sources: List[str], 
    output_dir: str, 
    verbose: bool = False,
    log_file: Optional[str] = None
) -> None:
    """Execute the extract command.
    
    Args:
        sources: List of source names to extract from
        output_dir: Directory to save output CSV files
        verbose: Whether to enable verbose logging
        log_file: Path to save log file
    """
    # Configure logging
    log_level = logging.DEBUG if verbose else logging.INFO
    setup_logging(level=log_level, log_file=log_file)
    
    # Create extractor
    extractor = StacItemExtractor(output_dir=output_dir)
    logger.info(f"Initialized extractor with output directory: {output_dir}")
    
    # Create a simple implementation of the MontyGeoCoder
    class SimpleGeoCoder(MontyGeoCoder):
        """A simple implementation of the MontyGeoCoder interface."""
        
        def get_geometry_by_country_name(self, country_name: str):
            """Get geometry for a country by name."""
            logger.info(f"Mock geocoding for country: {country_name}")
            # Return a simple point geometry for testing
            return {"type": "Point", "coordinates": [0, 0]}
            
        def get_geometry_from_admin_units(self, admin_units: List[str]):
            """Get geometry from a list of admin units."""
            logger.info(f"Mock geocoding for admin units: {admin_units}")
            # Return a simple point geometry for testing
            return {"type": "Point", "coordinates": [0, 0]}
            
        def get_geometry_from_iso3(self, iso3: str):
            """Get geometry for a country by ISO3 code."""
            logger.info(f"Mock geocoding for ISO3: {iso3}")
            # Return a simple point geometry for testing
            return {"type": "Point", "coordinates": [0, 0]}
            
        def get_iso3_from_geometry(self, geometry):
            """Get ISO3 code for a country based on a geometry."""
            logger.info(f"Mock reverse geocoding")
            # Return a mock ISO3 code
            return "XYZ"
    
    # Create an instance of our implementation
    geocoder = SimpleGeoCoder()
    
    # Process each source
    for source_name in sources:
        source_name = source_name.lower().strip()
        
        if source_name not in SOURCE_MAPPING:
            logger.error(f"Unknown source: {source_name}")
            continue
        
        if source_name not in SOURCE_CONFIGS:
            logger.error(f"No configuration available for source: {source_name}")
            continue
            
        logger.info(f"Processing source: {source_name}")
        
        # Get transformer and data source classes
        transformer_class, data_source_class = SOURCE_MAPPING[source_name]
        
        try:
            # Get source configuration
            config = SOURCE_CONFIGS[source_name].copy()  # Make a copy to avoid modifying the original
            
            # Create data source
            data_source = data_source_class(**config)
            
            # Create transformer with appropriate parameters
            if source_name == "ibtracs":
                transformer = transformer_class(data_source, geocoder)
            else:
                transformer = transformer_class(data_source)
            
            # Extract data, passing the source_name explicitly
            extractor.extract_from_transformer(transformer, source_name)
            
        except Exception as e:
            logger.error(f"Error processing {source_name}: {e}")
    
    # Save results
    logger.info("Saving results to CSV files")
    extractor.save_to_csv()
    logger.info("Done!")


def list_command() -> None:
    """Execute the list command."""
    print("Available sources:")
    for source in sorted(SOURCE_MAPPING.keys()):
        print(f"  - {source}")


def main() -> None:
    """Main entry point for the CLI."""
    # Check if running in a Jupyter notebook
    is_jupyter = 'ipykernel' in sys.modules
    
    if is_jupyter:
        # If in Jupyter, skip argparse and extract from all sources
        print("Running in Jupyter notebook environment")
        print("Extracting data from all available sources...")
        
        # Get all available sources
        sources = list(SOURCE_MAPPING.keys())
        print(f"Sources to process: {', '.join(sources)}")
        
        # Extract data from all sources
        output_dir = "output"
        extract_command(sources=sources, output_dir=output_dir, verbose=True)
    else:
        # Normal CLI execution
        parser = create_parser()
        args = parser.parse_args()
        
        if args.command == "extract":
            sources = [s.strip() for s in args.source.split(",")]
            extract_command(
                sources=sources,
                output_dir=args.output,
                verbose=args.verbose,
                log_file=args.log_file
            )
        elif args.command == "list":
            list_command()
        else:
            parser.print_help()
            sys.exit(1)


if __name__ == "__main__":
    main()

2025-04-23 07:14:44,461 - __main__ - INFO - Initialized extractor with output directory: output
2025-04-23 07:14:44,462 - __main__ - INFO - Processing source: emdat
2025-04-23 07:14:44,462 - __main__ - ERROR - Error processing emdat: [Errno 2] No such file or directory: '{"query": "select * from emdat limit 10"}'
2025-04-23 07:14:44,463 - __main__ - INFO - Processing source: desinventar
2025-04-23 07:14:44,463 - __main__ - INFO - Extracting data from desinventar transformer
2025-04-23 07:14:44,464 - __main__ - ERROR - Error processing desinventar: 'DesinventarTransformer' object has no attribute 'get_stac_items'
2025-04-23 07:14:44,464 - __main__ - INFO - Processing source: gdacs_flood
2025-04-23 07:14:44,465 - __main__ - INFO - Extracting data from gdacs_flood transformer
2025-04-23 07:14:44,466 - __main__ - ERROR - Error processing gdacs_flood: 'GDACSTransformer' object has no attribute 'get_stac_items'
2025-04-23 07:14:44,466 - __main__ - INFO - Processing source: gdacs_drought
2025

Running in Jupyter notebook environment
Extracting data from all available sources...
Sources to process: emdat, desinventar, gdacs_flood, gdacs_drought, gfd, gidd, glide, ibtracs, idu, pdc, usgs
